In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import root_scalar
from sympy.physics.wigner import wigner_3j

# Constants for 60CoCo(hcp) thermometer from "jres-88-175.pdf"
SPIN_I = 5
DELTA_M_MK = -6.0725  # Dipole splitting in mK 
P_K_MK = -0.0022      # Quadrupole coupling constant in mK 

# U_lambda * A_lambda coefficients for the 1332.5 keV transition
# Derived from U2=0.703731, A2=-0.597614 and U4=0.227128, A4=-1.069046
U2A2 = -0.420560
U4A4 = -0.242810

# Assume solid angle correction factors Q_lambda ~ 1 for a point source
Q2 = 0.9965
Q4 = 0.9886

def calculate_energy_levels(m):
    """Computes the energy levels Em/k for 60Co."""
    return -DELTA_M_MK * m + P_K_MK * (m**2 - (1/3) * SPIN_I * (SPIN_I + 1))

def calculate_B_lambda(T, lambda_val):
    """
    Computes the nuclear orientation parameter B_lambda(I, T) for a given temperature T (in mK).
    """
    m_values = np.arange(-SPIN_I, SPIN_I + 1)
    energies = calculate_energy_levels(m_values)
    
    # Boltzmann distribution populations
    exponents = np.exp(-energies / T)
    Z = np.sum(exponents)
    populations = exponents / Z
    
    B_lambda = 0
    for m, p_m in zip(m_values, populations):
        # Wigner 3-j symbol evaluation
        w3j = float(wigner_3j(SPIN_I, SPIN_I, lambda_val, m, -m, 0))
        term = ((-1)**(SPIN_I - m)) * np.sqrt((2 * lambda_val + 1) * (2 * SPIN_I + 1)) * w3j * p_m
        B_lambda += term
        
    return B_lambda

def theoretical_W0(T):
    """
    Computes the theoretical normalized directional intensity W(0) at temperature T (mK).
    """
    B2 = calculate_B_lambda(T, 2)
    B4 = calculate_B_lambda(T, 4)
    
    # P_lambda(cos(0)) = 1 for all lambda
    return 1 + (B2 * U2A2 * Q2) + (B4 * U4A4 * Q4)

def extract_counts(file_path, v_min, v_max):
    """
    Reads the CSV, skips corrupted rows, drops 'timestamp' if present, 
    and extracts total counts within the pulse height window (ROI).
    """
    try:
        # Standard for pandas >= 1.3.0
        df = pd.read_csv(file_path, on_bad_lines='skip')
    except TypeError:
        # Fallback for older pandas versions
        df = pd.read_csv(file_path, error_bad_lines=False)
    
    # Drop the 'timestamp' column safely if present
    df = df.drop(columns=['timestamp'], errors='ignore')
    
    # Filter by ROI window
    roi_events = df[(df['pulse_height_V'] >= v_min) & (df['pulse_height_V'] <= v_max)]
    return len(roi_events)

def main():
    # File paths for the two CSV files
    cold_file = "co60_15mK_front_cryostat.csv"
    warm_file = "new_co60_above_100K.csv"
    
    # Define the Region of Interest (ROI) for the 1332.5 keV photopeak in Volts
    # (These values must be calibrated to your specific detector's gain)
    V_MIN = 0.7
    V_MAX = 0.95
    
    # 1. Extract counts for cold and warm spectra
    C_c = extract_counts(cold_file, V_MIN, V_MAX)
    C_w = extract_counts(warm_file, V_MIN, V_MAX)
    
    # Print individual counts
    print(f"Cold Counts (C_c): {C_c}")
    print(f"Warm Counts (C_w): {C_w}")
    
    if C_w == 0:
        raise ValueError("Warm counts cannot be zero.")
        
    # 2. Calculate measured W(0)
    W0_measured = C_c / C_w
    print(f"Measured W(0): {W0_measured:.6f}")
    
    # 3. Solve for Temperature (T)
    # We define a residual function to find the root where theoretical_W0(T) == W0_measured
    def residual(T):
        return theoretical_W0(T) - W0_measured
        
    try:
        # Search for T in the useful range of 1.3 mK to 50 mK
        result = root_scalar(residual, bracket=[1.0, 100.0], method='brentq')
        T_mK = result.root
        print(f"Calculated Temperature: {T_mK:.3f} mK")
    except ValueError:
        print("Error: Measured W(0) falls outside the theoretical temperature bounds (1-100 mK).")

if __name__ == "__main__":
    main()

Cold Counts (C_c): 2215
Warm Counts (C_w): 1372
Measured W(0): 1.614431
Error: Measured W(0) falls outside the theoretical temperature bounds (1-100 mK).
